# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "dataset.csv"          
PROCESSED = Path("..") / "data" / "processed"
#TARGET = "target"                                            

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")
df.head()

,ID,AMT_INCOME_TOTAL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,TARGET,YEARS_BIRTH,YEARS_EMPLOYED,CODE_GENDER_DUMMY,FLAG_OWN_CAR_DUMMY,FLAG_OWN_REALTY_DUMMY,NAME_EDUCATION_TYPE_NUM,OCCUPATION_TYPE_GROUPED_Core staff,OCCUPATION_TYPE_GROUPED_Drivers,OCCUPATION_TYPE_GROUPED_Laborers,OCCUPATION_TYPE_GROUPED_Managers,OCCUPATION_TYPE_GROUPED_Others,OCCUPATION_TYPE_GROUPED_Pensioner,OCCUPATION_TYPE_GROUPED_Sales staff,OCCUPATION_TYPE_GROUPED_Working,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents
0,5008804,427500.0,1,0,0,2.0,1,32.890411,12.443836,1,1,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,5008805,427500.0,1,0,0,2.0,1,32.890411,12.443836,1,1,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,5008806,112500.0,0,0,0,2.0,0,58.832877,3.106849,1,1,1,2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,5008808,270000.0,0,1,1,1.0,0,52.356164,8.358904,0,0,1,2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,5008809,270000.0,0,1,1,1.0,0,52.356164,8.358904,0,0,1,2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


## 1. Split treino/teste

In [3]:
#`stratify` preserva a proporção das classes nos dois conjuntos.

from sklearn.model_selection import train_test_split

X = df.drop(columns=['TARGET', 'ID'])
y = df['TARGET']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## 2. Modelos candidatos

In [4]:
#Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from feature_engine.outliers import Winsorizer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import LinearSVC

In [5]:
cols_outliers = ['AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS', 'YEARS_EMPLOYED']

# Pipeline - Estrutura de cada modelo
pipelines = {
    "KNN": Pipeline([
        ("winsorizer", Winsorizer(
            capping_method='quantiles', 
            tail='right', 
            fold=0.01, 
            variables=cols_outliers
            )),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", KNeighborsClassifier())
    ]),
    "SVM": Pipeline([
        ("winsorizer", Winsorizer(
            capping_method='quantiles', 
            tail='right', 
            fold=0.01, 
            variables=cols_outliers
            )),
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(random_state=RANDOM_STATE,max_iter=5000, class_weight='balanced', dual="auto"))
    ]),
    "Decision_Tree": Pipeline([
        ("clf", DecisionTreeClassifier(
            random_state=RANDOM_STATE, 
            class_weight='balanced'))
    ]),
    "Random_Forest": Pipeline([
        ("clf", RandomForestClassifier(
            random_state=RANDOM_STATE, 
            n_jobs=-1, 
            class_weight='balanced'))
    ])
}

# 2. Dicionário de Grades de Hiperparâmetros, para testar os melhores parâmetros para cada modelo.

from scipy.stats import randint, uniform
param_grids = {
    "KNN": {
        "clf__n_neighbors": [5, 7, 11, 15],
        "clf__weights": ["uniform", "distance"]
    },
    "SVM": {
        "clf__C": [0.1, 1, 10],
        "clf__loss": ["hinge", "squared_hinge"]
    },
    "Decision_Tree": {
        "clf__max_depth": [3, 5, 8, 12, None],
        "clf__min_samples_split": [2, 5, 10],
        "clf__criterion": ["gini", "entropy"]
    },
    "Random_Forest": {
        "clf__n_estimators": randint(100, 500),
        "clf__max_depth": [5, 10, 15, None],
        "clf__max_features": ["sqrt", "log2"]
    }
}



In [6]:
# Dicionário de métricas a serem avaliadas
scoring_grid = {
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision',
    'roc_auc': 'roc_auc'
}

resultados_rs = []
modelos = {}

for nome_modelo, pipeline in pipelines.items():
        
    random_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grids[nome_modelo],
        n_iter=10,
        cv=5,
        scoring=scoring_grid,
        refit="f1",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1
    )
    
    random_search.fit(X_train, y_train)
    
    # Índice da combinação de hiperparâmetros vencedora
    best_idx = random_search.best_index_
    
    # Extrai o desempenho do melhor modelo para todas as métricas
    resultados_rs.append({
        "Modelo": nome_modelo,
        "F1-Score": random_search.cv_results_['mean_test_f1'][best_idx].round(4)*100,
        "Recall": random_search.cv_results_['mean_test_recall'][best_idx].round(4)*100,
        "Precision": random_search.cv_results_['mean_test_precision'][best_idx].round(4)*100,
        "ROC-AUC": random_search.cv_results_['mean_test_roc_auc'][best_idx].round(4)*100,
        "Melhores Parâmetros": random_search.best_params_
    })
    
    modelos[nome_modelo] = random_search.best_estimator_

# Tabela comparativa
df_grid_resultados = pd.DataFrame(resultados_rs).sort_values(by="F1-Score", ascending=False)
display(df_grid_resultados)

c:\Users\User\OneDrive - Universidade Estadual de Campinas\Documentos\GitHub\rm377793-postech-tech-challenge-fase2\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 8 candidates, totalling 40 fits


c:\Users\User\OneDrive - Universidade Estadual de Campinas\Documentos\GitHub\rm377793-postech-tech-challenge-fase2\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits


,Modelo,F1-Score,Recall,Precision,ROC-AUC,Melhores Parâmetros
3,Random_Forest,39.45,54.30,30.99,76.86,"{'clf__max_depth': None, 'clf__max_features': ..."
2,Decision_Tree,39.06,57.91,29.47,71.74,"{'clf__min_samples_split': 2, 'clf__max_depth'..."
0,KNN,37.98,41.48,35.04,74.26,"{'clf__weights': 'distance', 'clf__n_neighbors..."
1,SVM,20.94,50.45,13.21,54.18,"{'clf__loss': 'squared_hinge', 'clf__C': 0.1}"


## 4. Comparação

A tabela *'df_grid_resultados'* traz um resumo do desempenho de cada modelo utilizado neste projeto. 

- Observa-se que o **Random Forest** foi o que obteve melhor F1 Score (39,45%) e maior ROC-AUC (76,86%).
- Quanto ao modelo de **Árvore de Decisão**, que ficou como segundo colocado pela métrica F1 Score, cabe destacar o seguinte: comparado ao Random Forest, teve F1 Score 0,45 pp menor. Nesse sentido, apesar da maior complexidade do Random Forest, não se observa ganhos substanciais quando comparado à Árvore de Decisão. Outro ponto interessante é que a métrica de Recall é melhor aqui, capturando melhor os clientes que são mau pagadores.
- O **KNN** ficou como terceiro colocado pela métrica F1 Score. Olhando pela métrica de Recall, foi o modelo com pior desempenho - em termos simples, a cada 100 clientes, o modelo identifica corretamente ~42 como mau pagadores. 
- Por fim, o modelo **SVM** obteve o pior desempenho, não sendo capaz de capturar a complexidade dos dados. 


In [7]:
#Salvando o modelo para utilização no notebook 4

import joblib
from pathlib import Path

# Criação do diretório de destino na pasta 'models' do projeto
MODELS_DIR = Path("..") / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Salva o dicionário com os 4 pipelines otimizados
joblib.dump(modelos, MODELS_DIR / "modelos_credito.pkl")

['..\\models\\modelos_credito.pkl']